# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第14章 第二言語音韻研究 ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `ch14_pam_schematic.R`

対象章: 第14章「第二言語音韻研究」

PAM/PAM-L2が区別する同化パターンの3類型(Two Category, Single Category,
Uncategorized-Categorized)を、母語カテゴリーの音響空間上に2つの
第二言語音がどう位置づけられるかとして図示する模式図。
実測データではなく、理論的な予測を視覚化するための概念図である。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(patchwork)

native_cats <- tibble(
  cat   = c("母語カテゴリーA", "母語カテゴリーB"),
  lab   = c("母語\nカテゴリーA", "母語\nカテゴリーB"),
  xmin  = c(0.5, 3.5),
  xmax  = c(2.0, 5.0)
)

make_panel <- function(pattern, l2_sounds, title, note) {
  # 白黒印刷で母語カテゴリーA/Bが区別できるよう、濃い灰色(A)と薄い灰色(B)に
  # し、境界線も付ける。L2音は黒の三角で示し、ラベルは上下にずらして重ねない。
  l2_sounds <- l2_sounds |> mutate(ly = c(0.80, 0.62))
  ggplot() +
    geom_rect(data = native_cats, aes(xmin = xmin, xmax = xmax, ymin = 0, ymax = 1, fill = cat),
              colour = "grey20", linewidth = 0.4) +
    geom_point(data = l2_sounds, aes(x = x, y = 0.45), size = 4.5, shape = 17, colour = "black") +
    geom_segment(data = l2_sounds, aes(x = x, xend = x, y = 0.45, yend = ly - 0.05),
                 colour = "black", linewidth = 0.3) +
    geom_label(data = l2_sounds, aes(x = x, y = ly, label = label), size = 4.2,
               fontface = "bold", colour = "black", fill = "white", linewidth = 0.3) +
    geom_text(data = native_cats, aes(x = (xmin + xmax) / 2, y = 0.14, label = lab), size = 2.9, lineheight = 0.95,
              fontface = "bold", colour = c("white", "black")) +
    scale_fill_manual(values = c("母語カテゴリーA" = "#4D4D4D", "母語カテゴリーB" = "#C8C8C8"), guide = "none") +
    scale_x_continuous(limits = c(0, 5.5), breaks = NULL, name = NULL) +
    scale_y_continuous(limits = c(0, 1), breaks = NULL, name = NULL) +
    labs(title = title, caption = note) +
    theme(panel.grid = element_blank(), plot.title = element_text(size = 12, face = "bold"),
          plot.caption = element_text(size = 10, colour = "black", hjust = 0, lineheight = 1.05))
}

p_tc <- make_panel(
  "TC",
  tibble(x = c(1.2, 4.2), label = c("L2音1", "L2音2")),
  "Two Category (TC)",
  "2つのL2音が別々の母語カテゴリーに同化\n→ 弁別は容易と予測"
)

p_sc <- make_panel(
  "SC",
  tibble(x = c(1.0, 1.6), label = c("L2音1", "L2音2")),
  "Single Category (SC)",
  "2つのL2音が同じ母語カテゴリーに\n強く同化(良さの差は小さい)\n→ 弁別は困難と予測"
)

p_uc <- make_panel(
  "UC",
  tibble(x = c(2.7, 4.2), label = c("L2音1", "L2音2")),
  "Uncategorized-Categorized (UC)",
  "1つは範疇間の隙間に非範疇化、\nもう1つは明確に同化\n→ 弁別可能性は他要因に依存"
)

p <- p_tc + p_sc + p_uc + plot_layout(nrow = 1)
ggsave(here::here("assets", "figures", "ch14", "fig14-01-pam-assimilation.png"),
       p, width = 9, height = 3.9, dpi = 320)
ggsave(here::here("assets", "figures", "ch14", "fig14-01-pam-assimilation.pdf"),
       p, width = 9, height = 3.9, device = cairo_pdf)
cat("完了。図はassets/figures/ch14/に保存した。\n")


## `ch14_perception_production_correlation.R`

対象章: 第14章「第二言語音韻研究」

「産出側の測定と、知覚–産出の対応」節の作例。知覚成績と産出成績が
真には中程度に相関している(r = 0.6)場合でも、それぞれの測定に
独立な誤差が乗ると、観測される相関係数がどれだけ減衰するかを、
架空のデータで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n <- 100
true_r <- 0.6

# 2変量正規分布から、真の相関r=0.6を持つ潜在的な「知覚能力」「産出能力」を生成
latent <- MASS::mvrnorm(n, mu = c(0, 0),
                         Sigma = matrix(c(1, true_r, true_r, 1), nrow = 2))
true_perception <- latent[, 1]
true_production <- latent[, 2]

measurement_noise_sd <- 0.8   # 各課題の測定に独立に乗る誤差

observed_perception <- true_perception + rnorm(n, 0, measurement_noise_sd)
observed_production <- true_production + rnorm(n, 0, measurement_noise_sd)

cat("真の潜在能力間の相関(理論値): r =", true_r, "\n")
cat("真の潜在変数どうしの標本相関: r =",
    round(cor(true_perception, true_production), 3), "\n")
cat("測定誤差を含む観測値どうしの相関: r =",
    round(cor(observed_perception, observed_production), 3), "\n")


## `ch14_measurement_error_attenuation.R`

対象章: 第14章「第二言語音韻研究」

「個人差・経験変数のモデル化」節の作例。自己報告による経験変数
(LOR)に測定誤差(想起バイアス)が含まれる場合、真の経験変数を
使った回帰係数に比べて、自己報告値を使った回帰係数がどれだけ
減衰(過小評価)するかを、架空のデータで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n <- 150
true_lor_years <- runif(n, 0, 10)          # 真の居住期間(年)
true_effect <- -3                          # LOR1年あたりの知覚困難度の真の減少

# 自己報告値には、想起バイアスによる測定誤差が乗る(標準偏差1.5年)
self_reported_lor <- true_lor_years + rnorm(n, 0, 1.5)

outcome <- 50 + true_effect * true_lor_years + rnorm(n, 0, 8)

m_true <- lm(outcome ~ true_lor_years)
m_self_reported <- lm(outcome ~ self_reported_lor)

cat("真のLORを予測変数とした場合の係数:", round(coef(m_true)[2], 2), "\n")
cat("自己報告LOR(測定誤差あり)を予測変数とした場合の係数:",
    round(coef(m_self_reported)[2], 2), "\n")
cat("減衰率:", round((1 - coef(m_self_reported)[2] / coef(m_true)[2]) * 100, 1), "%\n")


## `ch14_worked_example.R`

対象章: 第14章「第二言語音韻研究」通しの作例（第11章の同定課題の枠組みを応用）

「発音訓練研究とHVPT」節で述べた、高変動音素訓練の効果(訓練前後の
知覚改善、および新規話者・語彙への般化)を、架空のデータでシミュレートする。
実測データではない。刺激は第11章と同じくVOT連続体を想定するが、
ここでは日本語母語話者による英語/r/-/l/の同定課題を模した設定とする。

設計:
  同一の仮想学習者20名について、訓練前(pre)・訓練後(post)・
  訓練で使わなかった新規話者/語彙での般化テスト(generalization)の
  3条件で同定課題を行わせる。訓練前は境界が不安定(傾きが緩やか)で
  あるのに対し、訓練後は傾きが急になる(範疇知覚がより明確になる)と
  設定し、般化条件では訓練後の改善の一部が保持されるが完全ではない
  ように設定する。

統計的な扱い: 条件ごとに別々のモデルを当てるのではなく、条件を予測変数に
含めた単一のモデル(step_c * condition)を当て、条件間の傾きの差を
交互作用項として直接検定する。傾きの急峻化だけをもって「学習が改善した」
とは判断せず、刺激連続体の両端(範疇的にもっとも明確な刺激)での正答率も
あわせて確認し、境界の位置が正しいカテゴリーに対応しているかを検証する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260904)

n_subjects <- 20
steps      <- seq(0, 80, by = 10)
n_reps     <- 10
true_boundary <- 35   # /r/-/l/の物理的な境界と想定する値(このステップ未満が正しく/r/)

pop <- tibble(
  condition = c("pre", "post", "generalization"),
  slope_mean = c(0.09, 0.24, 0.19),     # 訓練前は緩やか、訓練後は急峻、般化はその中間
  boundary_mean = c(38, 33, 34)
)

subj_re <- tibble(
  subject = sprintf("s%02d", 1:n_subjects),
  slope_re    = rnorm(n_subjects, 0, 0.03),
  boundary_re = rnorm(n_subjects, 0, 5)
)

d <- pop |>
  tidyr::crossing(subj_re) |>
  mutate(slope = pmax(slope_mean + slope_re, 0.02),
         boundary = boundary_mean + boundary_re) |>
  tidyr::crossing(step = steps) |>
  tidyr::crossing(rep = 1:n_reps) |>
  mutate(
    logit_p = slope * (step - boundary),
    p_resp  = plogis(logit_p),
    response = rbinom(n(), 1, p_resp)
  ) |>
  select(subject, condition, step, rep, response) |>
  mutate(
    condition = factor(condition, levels = c("pre", "post", "generalization")),
    step_c    = as.numeric(scale(step, scale = FALSE))
  )

step_mean <- mean(d$step)

# 条件を予測変数に含めた単一モデル。conditionはpreを参照水準とする
# 処理コーディング(第10章参照)なので、step_c:conditionの係数がそのまま
# 「pre条件と比べた傾きの差」の検定になる。
m <- glmer(response ~ step_c * condition + (1 + step_c | subject),
           data = d, family = binomial,
           control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5)))

cat("=== 単一モデルの固定効果(条件をpre基準として) ===\n")
print(summary(m)$coefficients)

fx <- fixef(m)
slope_pre   <- fx["step_c"]
slope_post  <- fx["step_c"] + fx["step_c:conditionpost"]
slope_gen   <- fx["step_c"] + fx["step_c:conditiongeneralization"]

# post条件をconditionの参照水準に張り替えて再当てはめし、post対generalizationの
# 傾きの差についても、単一モデルの交互作用項として直接検定する
d_relevel <- d |> mutate(condition = relevel(condition, ref = "post"))
m_post_ref <- glmer(response ~ step_c * condition + (1 + step_c | subject),
                     data = d_relevel, family = binomial,
                     control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5)))
coef_post_ref <- summary(m_post_ref)$coefficients

cat("\n=== 傾きの交互作用検定(単一モデル内) ===\n")
cat("post - pre の傾きの差:", round(fx["step_c:conditionpost"], 3),
    " z =", round(summary(m)$coefficients["step_c:conditionpost", "z value"], 2),
    " p =", signif(summary(m)$coefficients["step_c:conditionpost", "Pr(>|z|)"], 3), "\n")
cat("generalization - pre の傾きの差:", round(fx["step_c:conditiongeneralization"], 3),
    " z =", round(summary(m)$coefficients["step_c:conditiongeneralization", "z value"], 2),
    " p =", signif(summary(m)$coefficients["step_c:conditiongeneralization", "Pr(>|z|)"], 3), "\n")
cat("generalization - post の傾きの差:", round(coef_post_ref["step_c:conditiongeneralization", "Estimate"], 3),
    " z =", round(coef_post_ref["step_c:conditiongeneralization", "z value"], 2),
    " p =", signif(coef_post_ref["step_c:conditiongeneralization", "Pr(>|z|)"], 3), "\n")

results <- tibble(
  condition = c("pre", "post", "generalization"),
  slope     = c(slope_pre, slope_post, slope_gen)
) |>
  mutate(
    intercept = c(fx["(Intercept)"],
                  fx["(Intercept)"] + fx["conditionpost"],
                  fx["(Intercept)"] + fx["conditiongeneralization"]),
    boundary  = step_mean - intercept / slope
  )
print(results)

# 傾きの急峻化だけでは「学習が改善した」ことを意味しない。境界が正しい
# カテゴリー(true_boundary)の近くに位置し、連続体の両端で高い正答率が
# 得られているかも、あわせて確認する。
acc_at_ends <- d |>
  filter(step %in% c(min(steps), max(steps))) |>
  group_by(condition, step) |>
  summarise(prop_response = mean(response), .groups = "drop") |>
  mutate(prop_correct = if_else(step < true_boundary, 1 - prop_response, prop_response))

cat("\n=== 刺激連続体両端での正答率(境界の妥当性の確認) ===\n")
print(acc_at_ends |> select(condition, step, prop_correct))

curve_df <- purrr::map_dfr(seq_len(nrow(results)), function(i) {
  tibble(condition = results$condition[i],
         step = seq(min(steps), max(steps), length.out = 200)) |>
    mutate(prop = plogis(results$intercept[i] + results$slope[i] * (step - step_mean)))
})

cond_labels <- c(pre = "訓練前", post = "訓練後", generalization = "般化テスト")
curve_df <- curve_df |> mutate(condition_ja = factor(cond_labels[as.character(condition)], levels = cond_labels))

p <- ggplot(curve_df, aes(step, prop, colour = condition_ja, linetype = condition_ja)) +
  geom_line(linewidth = 1) +
  scale_colour_manual(values = c("訓練前" = "#8C8C8C", "訓練後" = "#34518A", "般化テスト" = "#A8681F")) +
  scale_linetype_manual(values = c("訓練前" = "solid", "訓練後" = "22", "般化テスト" = "42")) +
  scale_y_continuous(labels = scales::percent, limits = c(0, 1)) +
  labs(x = "刺激ステップ", y = "「学習対象カテゴリー」反応の割合", colour = NULL, linetype = NULL,
       caption = "シミュレーションデータ。scripts/R/ch14_worked_example.R より生成。")
save_fig(p, "fig14-02-hvpt-curves", chapter = "ch14")

cat("\n傾きの変化(訓練前→訓練後):",
    round(results$slope[results$condition == "pre"], 3), "→",
    round(results$slope[results$condition == "post"], 3), "\n")
cat("般化テストでの傾き:", round(results$slope[results$condition == "generalization"], 3),
    "(訓練後の改善のうち",
    round(100 * (results$slope[results$condition == "generalization"] - results$slope[results$condition == "pre"]) /
            (results$slope[results$condition == "post"] - results$slope[results$condition == "pre"]), 0),
    "%を保持)\n")
cat("完了。図は assets/figures/ch14/ に保存した。\n")


## `ch14_three_dimension_dissociation.R`

対象章: 第14章「第二言語音韻研究」

「accentedness・comprehensibility・intelligibilityの区別」節の作例。
3つの構成概念が部分的にしか相関しない(独立した情報を持つ)場合、
「訛りは強いが理解は容易」「訛りは弱いが理解に労力を要する」といった
話者が、どれだけの割合で実際に存在しうるかを、架空のデータで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n <- 200
accentedness <- rnorm(n, 50, 15)
# comprehensibilityはaccentednessと中程度に相関しつつ、独立な要因も持つ
comprehensibility <- 0.5 * accentedness + rnorm(n, 25, 15)

d <- tibble(accentedness, comprehensibility)

cat("accentednessとcomprehensibilityの相関: r =",
    round(cor(accentedness, comprehensibility), 2), "\n")

# 「訛りは強い(上位25%)が理解は容易(下位25%、低いほど理解しやすいと仮定)」話者
strong_accent <- accentedness > quantile(accentedness, 0.75)
easy_comprehend <- comprehensibility < quantile(comprehensibility, 0.25)
n_dissociated <- sum(strong_accent & easy_comprehend)

cat("訛りが強い(上位25%)話者数:", sum(strong_accent), "\n")
cat("そのうち理解が容易(comprehensibility下位25%)な話者数:", n_dissociated,
    "(", round(n_dissociated / sum(strong_accent) * 100, 1), "%)\n")
